# BERT (DistilBERT) Sentiment Classifier

Fine-tuning DistilBERT for IMDB sentiment classification using HuggingFace Transformers.

## Imports

In [ ]:
import pandas as pd
import torch
import os
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding
from datasets import Dataset

## Configuration

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
SAVE_DIR = "models/bert_sentiment"
MAX_LENGTH = 256
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

# For quick verification, use a subset. Set to None for full dataset.
N_SAMPLES = 2000  # Set to None for full training

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## Load Tokenizer and Data

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Loading data...")
train_df = pd.read_csv("data/train.csv").dropna(subset=['review'])
val_df = pd.read_csv("data/val.csv").dropna(subset=['review'])

if N_SAMPLES:
    train_df = train_df.sample(N_SAMPLES, random_state=42)
    val_df = val_df.sample(N_SAMPLES // 2, random_state=42)

print(f"Train size: {len(train_df)}")
print(f"Val size: {len(val_df)}")

## Prepare HuggingFace Datasets

In [ ]:
def tokenize_function(examples):
    return tokenizer(examples["review"], padding="max_length", truncation=True, max_length=MAX_LENGTH)

# Convert to HuggingFace Datasets
train_hf = Dataset.from_pandas(train_df[['review', 'sentiment']])
val_hf = Dataset.from_pandas(val_df[['review', 'sentiment']])

# Encode labels
train_hf = train_hf.map(lambda x: {"label": 1 if x["sentiment"] == 'positive' else 0})
val_hf = val_hf.map(lambda x: {"label": 1 if x["sentiment"] == 'positive' else 0})

# Tokenize
print("Tokenizing datasets...")
train_dataset = train_hf.map(tokenize_function, batched=True)
val_dataset = val_hf.map(tokenize_function, batched=True)

## Define Metrics

In [ ]:
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='binary')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

## Initialize Model and Trainer

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=SAVE_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_dir='./logs',
    logging_steps=100
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

## Train the Model

In [ ]:
print("Starting training...")
trainer.train()

## Save the Model

In [ ]:
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Model saved to {SAVE_DIR}")

## Quick Prediction Test

In [ ]:
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    pred_label_id = torch.argmax(probs, dim=1).item()
    sentiment = "positive" if pred_label_id == 1 else "negative"
    confidence = probs[0][pred_label_id].item()
    return sentiment, confidence

text = "This movie was absolutely wonderful!"
sentiment, confidence = predict(text)
print(f"Text: {text}")
print(f"Prediction: {sentiment} (confidence: {confidence:.4f})")